In [ ]:
transactions_df = spark.table("workspace.default.bronze_transactions")
users_df = spark.table("workspace.default.bronze_users")
cards_df = spark.table("workspace.default.bronze_cards")

display(transactions_df.limit(3))
display(users_df.limit(3))
display(cards_df.limit(3))

In [ ]:
# convert amount string to numeric
from pyspark.sql.functions import *
transactions_df = transactions_df.withColumn("amount", regexp_replace(col("amount"),"[$]","").cast("decimal(10,2)"))
transactions_df.select("amount").show(3)
transactions_df.select("amount").printSchema()

In [ ]:
transactions_df.select("time").show(10)

In [ ]:
# Storing time value in time column instead of timestamp
transactions_df = transactions_df.withColumn("time",date_format(col("time"),"HH:mm:ss"))
transactions_df.select("time").show(5)

In [ ]:
# Timestamp Column
transactions_df = transactions_df.withColumn("transaction_timestamp", to_timestamp(concat_ws(" ",concat_ws("-",col("year"),lpad(col("month"),2,"0"),lpad(col("day"),2,"0")),date_format(col("time"),"HH:mm:ss")), "yyyy-MM-dd HH:mm:ss")) #lpad is used to add leading zeros to month and day and 2 means fixing to 2 digits eg 5 - 05

transactions_df.select("transaction_timestamp").show(3)

In [ ]:
# Identify Duplicates records
duplicate_rows = transactions_df.groupBy(transactions_df.columns).agg(count("*").alias("count")).filter(col("count")>1)
duplicate_rows.show()

In [ ]:
transactions_df = transactions_df.drop_duplicates()
print("no of rows after removing duplicates",transactions_df.count())

In [ ]:
#Grouping to find type of error counts
transactions_df.groupBy(col("errors")).agg(count("*").alias("count")).show()

In [ ]:
# Replacing null vales of col "zip","merchant_state","errors"
transactions_df = transactions_df.withColumn("zip",when(col("zip").isNull(),lit(0)).otherwise(col("zip"))).withColumn("merchant_state",when(col("merchant_state").isNull(),lit("UNKNOWN")).otherwise(col("merchant_state"))).withColumn("errors",when(col("errors").isNull(),lit("NO_ERRORS")).otherwise(col("errors")))

In [ ]:
transactions_df = transactions_df.withColumnRenamed("zip","zip_code")

In [ ]:
transactions_df.show(5)

In [ ]:
transactions_df.printSchema()

In [ ]:
transactions_df = transactions_df.withColumnRenamed("merchant_name","merchant_id")

In [ ]:
#Checking if amount has negative value
transactions_df.select(min("amount").alias("min_amount"),max("amount").alias("max_amount")).show()

In [ ]:
transactions_df.filter(col("amount")<=0).count()

In [ ]:
#Creating column is_refund for negative amounts
transactions_df = transactions_df.withColumn("is_refund",when(col("amount")<0,1).otherwise(0))

transactions_df.groupBy(col("is_refund")).count().show()

In [ ]:
#Creating column transaction_date
transactions_df = transactions_df.withColumn("transaction_date",to_date(col("transaction_timestamp")))

transactions_df.select("transaction_date","transaction_timestamp").show(5)

In [ ]:
#Creating column transaction_status for column have no errors is success and other faliure
transactions_df = transactions_df.withColumn("transaction_Status", when (col("errors")== "NO_ERRORS", "SUCCESS").otherwise("FAILURE"))


In [ ]:
#Creating numeric Fraud Flag(We have is_fraud) - Yes/No
transactions_df = transactions_df.withColumn("fraud_flag", when(col("is_fraud")== "Yes", 1).otherwise(0))

In [ ]:
#Creating transaction Hour
transactions_df = transactions_df.withColumn("transaction_hour", hour(col("transaction_timestamp")))

In [ ]:
#Creating Day of week
transactions_df = transactions_df.withColumn("day_of_week",date_format(col("transaction_timestamp"),"EEEE"))
transactions_df.select("day_of_week").show(5)

In [ ]:
transactions_df.printSchema()

In [ ]:
#Renaming of columns 
transactions_df = (
    transactions_df
    .withColumnRenamed("user", "user_id")
    .withColumnRenamed("card", "card_id")
    .withColumnRenamed("year", "transaction_year")
    .withColumnRenamed("month", "transaction_month")
    .withColumnRenamed("day", "transaction_day")
    .withColumnRenamed("use_chip", "payment_method")
    .withColumnRenamed("errors", "error_description")
    .withColumnRenamed("is_fraud", "fraud_status")
)

In [ ]:
from pyspark.sql.functions import sha2, concat_ws

transactions_df = transactions_df.withColumn(
    "transaction_id",
    sha2(
        concat_ws(
            "||",
            col("user_id").cast("string"),
            col("card_id").cast("string"),
            col("transaction_timestamp").cast("string"),
            col("amount").cast("string"),
            col("merchant_id").cast("string"),
            col("mcc").cast("string"),
            col("error_description").cast("string")
        ),
        256
    )
)

In [ ]:
transactions_df.select("transaction_id").distinct().count()

In [ ]:
Duplicate_transaction_id = transactions_df.groupBy(col("transaction_id")).agg(count("*").alias("count")).filter(col("count")>1)
Duplicate_transaction_id.show()

In [ ]:
transactions_df.join(
        Duplicate_transaction_id.select("transaction_id"),
        on="transaction_id",
        how="inner"
    ).orderBy("transaction_id").show()

In [ ]:
transactions_df.printSchema()

In [ ]:
transactions_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_transactions")

In [ ]:
spark.table("workspace.default.silver_transactions").show(5)

USERS_DF Transformation

In [ ]:
users_df = spark.table("workspace.default.bronze_users")

In [ ]:
users_df.show(2)

In [ ]:
print(users_df.columns)

In [ ]:
#Convert to Numeric Column due to symbols like "$", "," making it String
from pyspark.sql.functions import regexp_replace, col

currency_columns = [
    "per_capita_income_zipcode",
    "yearly_income_person",
    "total_debt"
]

for column_name in currency_columns:
    users_df = users_df.withColumn(
        column_name,
        regexp_replace(col(column_name), "[$,]", "").cast("decimal(12,2)")
    )

users_df.select(
    "per_capita_income_zipcode",
    "yearly_income_person",
    "total_debt"
).show(3)

In [ ]:
from pyspark.sql.functions import *
users_df = users_df.withColumn("apartment", when(col("apartment").isNull(), lit(0)).otherwise(col("apartment")))
users_df.select("apartment").show(5)

In [ ]:
users_df.select(min('per_capita_income_zipcode').alias("minimum_per_capita"), max('per_capita_income_zipcode').alias("maximum_per_capita"),min('yearly_income_person').alias("minimum_income"),max('yearly_income_person').alias("maximum_income"), min("total_debt").alias("min_debt"),
    max("total_debt").alias("max_debt")
).show()

In [ ]:
# Create Income Band
users_df = users_df.withColumn("income_band", when(col("yearly_income_person") <50000 ,"Low Income").when(col("yearly_income_person") <100000 ,"Medium_Income").when(col("yearly_income_person")<200000,"High_Income").otherwise("Very High Income"))

In [ ]:
# Debt to income ratio
users_df = users_df.withColumn("debt_to_income_ratio",(col("total_debt")/col("yearly_income_person")).cast("decimal(10,2)"))
users_df.select("debt_to_income_ratio","total_debt","yearly_income_person").show(5)

In [ ]:
# create_credit_profile
users_df = users_df.withColumn("credit_profile", when(col("fico_score") >= 750, "Excellent").when(col("fico_score") >= 700, "Good").when(col("fico_score") >= 650, "Fair").otherwise("Poor"))
users_df.groupBy("credit_profile").count().show()

In [ ]:
# Create column Segment
users_df = users_df.withColumn("customer_segment", 
                              when((col("yearly_income_person")>= 150000) & (col("fico_score") >= 750 ), "Premium")
                              .when((col("yearly_income_person")>= 100000) & (col("fico_score") >= 700), "Affulent")
                              .when(
        (col("yearly_income_person") >= 50000),
        "Standard"
    )
    .otherwise("Basic")
)

In [ ]:
users_df.select(
    "yearly_income_person",
    "income_band",
    "total_debt",
    "debt_to_income_ratio",
    "fico_score",
    "credit_profile",
    "customer_segment"
).show(5, False)

In [ ]:
users_df = users_df.withColumnRenamed("person","customer_name").withColumnRenamed("current_age","age").withColumnRenamed("per_capita_income_zipcode","per_capita_income").withColumnRenamed("yearly_income_person","annual_income").withColumnRenamed("zipcode","zip_code")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window_spec = Window.orderBy(F.monotonically_increasing_id())

users_df = users_df.withColumn(
    "user_id",
    F.row_number().over(window_spec) - 1
)

In [ ]:
users_df.select("user_id", "customer_name").show(10)

In [ ]:
from pyspark.sql import functions as F

card_counts = (
    cards_df
    .groupBy("user")
    .count()
    .withColumnRenamed("user", "user_id")
)

users_card_check = (
    users_df
    .join(card_counts, "user_id", "left")
    .withColumn("actual_card_count", F.coalesce(F.col("count"), F.lit(0)))
)

users_card_check.filter(
    F.col("actual_card_count") != F.col("num_credit_cards")
).count()

In [ ]:
users_df.printSchema()
print("users columns:",len(users_df.columns))

In [ ]:
users_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.default.silver_users")

In [ ]:
transactions_df.show(5)

In [ ]:
users_df.show(5)

In [ ]:
cards_df = spark.table("workspace.default.bronze_cards")
display(cards_df.limit(5))

In [ ]:
#Convert Credit limit to Numeric from String
from pyspark.sql.functions import *
cards_df = cards_df.withColumn("credit_limit",regexp_replace(col("credit_limit"),"[$,]","").cast("decimal(12,2)") )

In [ ]:
#Convert acct_open_date to date
cards_df = cards_df.withColumn("acct_open_date", to_date(concat(lit("01/"),col("acct_open_date")),"dd/MM/yyyy"))

In [ ]:
cards_df.select("acct_open_date").show(3)

In [ ]:
#Rename expires to expiry_date
cards_df = cards_df.withColumnRenamed("expires", "expiry_date")

In [ ]:
cards_df.select("expiry_date").show(5)

In [ ]:
#Convert expiry_date to date
#As we see expire date don't have day no so expilictly taking 01 and adding the comlete date
cards_df= cards_df.withColumn("expiry_date",to_date(concat(lit("01/"),col("expiry_date")),"dd/MM/yyyy"))

In [ ]:
cards_df.select("expiry_date").show(3)

In [ ]:
cards_df.select("has_chip").distinct().show()

In [ ]:
#Standardize has_chip column and card_on_dark_web
cards_df = cards_df.withColumn("has_chip", upper(trim(col("has_chip")))).withColumn("card_on_dark_web",upper(trim(col("card_on_dark_web"))))

In [ ]:
#Creating new column card_status - AT_RISK - on dark web otherwise ACTIVE
cards_df = cards_df.withColumn("cards_status",when(col("card_on_dark_web")=="YES", "AT_RISK").otherwise("ACTIVE"))

In [ ]:
#Creating New column card_age_years to find how old the card is
cards_df = cards_df.withColumn("card_age_years", floor(months_between(current_date(), col("acct_open_date"))/12))


In [ ]:
cards_df.show(3)

In [ ]:
#Create new column Credit limit band
cards_df= cards_df.withColumn("credit_limit_band",when(col("credit_limit")<5000 , "Low").when(col("credit_limit")<15000, "Medium").when(col("credit_limit")<30000,"High").otherwise("Very High"))

In [ ]:
cards_df.printSchema()

In [ ]:
#Data Quality Check
#Null Check
cards_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in cards_df.columns]).show()


In [ ]:
cards_df.count()

In [ ]:
cards_df.distinct().count()

In [ ]:
cards_df.select("user","card_index").distinct().count()

In [ ]:
cards_df.select(
    "credit_limit"
).summary(
    "count",
    "min",
    "max").show()

In [ ]:
cards_df.groupBy("card_brand").count().show()
cards_df.groupBy("card_type").count().show()

In [ ]:
#Writing final silver table
cards_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.default.silver_cards")

In [ ]:
users_df.printSchema()